In [ ]:
"Authors: Dominik Westphal and Jan Nöller"

from collections import deque
from pysat.solvers import Cadical195
import prefixtree

def make_vars(all_nodes, N, symbols):
    """
    Maps (x, node, color) and (d, color, symbol, color2) tuples to unique
    1-indexed SAT variable numbers.
    """
    var = {}
    counter = 1
    for v in all_nodes:
        for c in range(N):
            var[('x', v, c)] = counter
            counter += 1
    for c in range(N):
        for sym in symbols:
            for c2 in range(N):
                var[('d', c, sym, c2)] = counter
                counter += 1
    return var


def bfs_sorted(all_nodes, edges, root=0):
    """
    Returns all_nodes in BFS order starting from root, following the
    given edges
    """
    adjacency: dict[int, list[int]] = {v: [] for v in all_nodes}
    for (frm, to, _) in edges:
        adjacency[frm].append(to)

    visited = {root}
    order = [root]
    queue = deque([root])
    while queue:
        v = queue.popleft()
        for w in adjacency.get(v, []):
            if w not in visited:
                visited.add(w)
                order.append(w)
                queue.append(w)
    return order


def try_solve(edges, labels, all_nodes, N, symbols=None, root=0):
    """
    Tries to find a DFA with N states consistent with the labelled APTA,
    following Heule & Verwer's "Exact DFA Identification Using SAT Solvers"
    encoding, generalized to an arbitrary alphabet and an arbitrary number
    of distinct labels.

    Parameters:
        edges: list of (from, to, symbol) APTA transitions.
        labels: dict mapping (a subset of) APTA states to a label value.
            Two states with different label values are NEVER allowed to
            be merged into the same DFA state. Two states with the SAME
            label value may be merged. Any number of distinct labels works.
        all_nodes: iterable/set of all APTA states.
        N: candidate number of DFA states.
        symbols: alphabet (iterable). If None, inferred from edges.
        root: root/start state of the APTA (default 0).
        solver_class: pysat solver class to use (default Cadical195).

    Returns:
        None if UNSAT (try a larger N), otherwise:
            (coloring, transitions, state_labels)
        - coloring: dict APTA node -> DFA color (0..N-1)
        - transitions: dict (color, symbol) -> color2
        - state_labels: dict DFA color -> label, for every color that at
          least one labelled APTA node maps to (consistent by construction)
    """
    if symbols is None:
        symbols = sorted({sym for (_, _, sym) in edges})

    var = make_vars(all_nodes, N, symbols)
    solver = Cadical195()

    # Group labelled nodes by their label value
    nodes_by_label: dict = {}
    for v, lab in labels.items():
        nodes_by_label.setdefault(lab, []).append(v)
    distinct_labels = list(nodes_by_label.keys())

    # 1. Each node gets at least one color
    for v in all_nodes:
        solver.add_clause([var[('x', v, c)] for c in range(N)])

    # 2. States with different labels cannot share a color
    #    (states with the same label CAN share a color)
    for i in range(len(distinct_labels)):
        for j in range(i + 1, len(distinct_labels)):
            for v in nodes_by_label[distinct_labels[i]]:
                for u in nodes_by_label[distinct_labels[j]]:
                    for c in range(N):
                        solver.add_clause([-var[('x', v, c)], -var[('x', u, c)]])

    # 3. Transition consistency
    for (v, w, sym) in edges:
        for c in range(N):
            for c2 in range(N):
                solver.add_clause([-var[('x', v, c)], -var[('x', w, c2)], var[('d', c, sym, c2)]])

    # 4. Transition function well-defined (exactly one target per (color, symbol))
    for c in range(N):
        for sym in symbols:
            solver.add_clause([var[('d', c, sym, c2)] for c2 in range(N)])
            for c2 in range(N):
                for c3 in range(c2 + 1, N):
                    solver.add_clause([-var[('d', c, sym, c2)], -var[('d', c, sym, c3)]])

    # 5. Root gets color 0 (symmetry breaking)
    solver.add_clause([var[('x', root, 0)]])

    # 6. BFS symmetry breaking
    bfs_order = bfs_sorted(all_nodes, edges, root=root)
    for i, v in enumerate(bfs_order):
        for c in range(i + 1, N):
            solver.add_clause([-var[('x', v, c)]])

    # 7. Each node gets at most one color
    for v in all_nodes:
        for c in range(N):
            for c2 in range(c + 1, N):
                solver.add_clause([-var[('x', v, c)], -var[('x', v, c2)]])

    if not solver.solve():
        return None,None,None

    model = set(solver.get_model())

    coloring = {}
    for v in all_nodes:
        for c in range(N):
            if var[('x', v, c)] in model:
                coloring[v] = c
                break

    transitions = {}
    for c in range(N):
        for sym in symbols:
            for c2 in range(N):
                if var[('d', c, sym, c2)] in model:
                    transitions[(c, sym)] = c2
                    break

    state_labels = {}
    for v, lab in labels.items():
        state_labels[coloring[v]] = lab

    return coloring, transitions, state_labels

def get_output(coloring, transitions, labels):
    """
    Converts a SAT-solved coloring/transition assignment into a minimal
    DFA description.

    Args:
        coloring: dict APTA node -> DFA state (color).
        transitions: dict (state, symbol) -> state2.
        labels: dict mapping (some) APTA nodes to a label value
            (works for any number of distinct label values).

    Returns:
        minEdges: list of (state, state2, symbol) transitions, sorted by
            (state, symbol) for determinism/reproducibility.
        state_labels: dict mapping DFA state -> label, for every state
            that at least one labelled APTA node maps to.
    """
    minEdges = []
    for (c, sym), c2 in sorted(transitions.items()):
        minEdges.append((c, c2, sym))

    state_labels: dict[int, object] = {}
    for v, lab in labels.items():
        c = coloring[v]
        state_labels[c] = lab

    return minEdges, state_labels


def create_MUB_promise_problem(d: int) -> list[set]:
    labelled_word_list = [set([]) for i in range(d)]
    for j in range (d+2):
        for k in range(d+2):
            for l in range(d+2):
                for m in range(d+2):
                    for r in range(d+2):
                        word = "s"
                        word += "x" * j
                        word += "s"
                        word += "x" * k
                        word += "s"
                        labelled_word_list[(2*k+j)%d].add(word)
                        word += "x" * r
                        word += "s"
                        word += "x" * l
                        word += "s"
                        word += "x" * m
                        word += "s"
                        labelled_word_list[(2*k+j+2*m+l)%d].add(word)
            word = ""
            word += "s"*j
            word += "x" * k
            word += "s" * (d-j)
            labelled_word_list[(k*j)%d].add(word)  
    labelled_word_list[0].add("")
    labelled_word_list[0].add("x")  
    return labelled_word_list

def check_solvability(d : int, promise_prob: list[set], upper = None):    
    if upper == None:
        upper = d**2
    print(upper)
    if len(promise_prob) != d:
        return 0
    for i in range(len(promise_prob)):
        print(promise_prob[i])
    
    edges,labels,states = prefixtree.build_apta(d,promise_prob)
    symbols = ['x','s']
    for n in range(d,upper):
        coloring,transitions,state_labels = try_solve(edges,labels,states,n,symbols)
        if(coloring == None):
            continue
        min_edges, lab = get_output(coloring,transitions,state_labels)
        print(min_edges)
        print(lab)




In [ ]:
PP = create_MUB_promise_problem(3)
check_solvability(3,PP)


9
{'', 'sxsxxxxssxsxxxxs', 'sxxxsxxsxxxssxs', 'sxxxxsxxsxxxsxss', 'sxsxssxsxs', 'sxsxxxxsxsss', 'sxxsxxsxxxsxxxsxxxs', 'sxxxxsxxsxxxsxxxsxxs', 'ssxxxsxxsxxxss', 'sxxxsxssxxsxs', 'sxsxxsxsxxxsxxs', 'sxxxsxxxssxsxs', 'sxsxxxxsxxxxsxxxxsxxxxs', 'sxxxxsssxxxxsxxs', 'sxxxxsxssxxxss', 'sxxxxsxxxsxxxxssxxxxs', 'sxxxxsxxxssxxxsxxxxs', 'sssxxxsxxxxsxxxxs', 'sxxsxxsxsxxxsxxxs', 'sxxsxxsxxxsxsxxxxs', 'sxxsxxssxxsxxs', 'sxxssxxxsxxsxxxxs', 'sxxssxxxsxsxxxs', 'sxxsxxxxsxxsxxsxxxs', 'sxsxxxxsxxsxxsxxs', 'sxxxsxxxsxxsxxxsxxxs', 'sxxxxssxsxxss', 'sxxxsxxssxxss', 'sxxxsxxxssxxxss', 'sxxxxsssxxxsxxxxs', 'sxxxxssxsxxsxxxs', 'ssxssxsxxxs', 'ssxxsxsxsxxs', 'sxxsxxxssxxsxxxxs', 'sxxsxxxsxsxxxxsxxxs', 'sxxxxsxssxsxxxxs', 'sxxxxsxsxxsxxxss', 'sxxxxsxsxxxxsxxxxsxxxxs', 'sxxxxsxxsxxxxsxxxxsxxxs', 'sxxxsxxxsxxxxsxxxxsxxxxs', 'sxxxxsxsxsxxxxsxxxxs', 'sxsxxxsxxsxxss', 'sxxxxsxxxsxxsxsxxs', 'ssxxxxsxxsxxxsxxs', 'sxssxsxsxxs', 'sssxsxxxss', 'sxxssxxsxsxxxs', 'sxxssxsxxsxxxxs', 'sxxxsxxxsxxsxsxs', 'sssxxxxsxsxs', 'sx